# Antardrishti — accuracy per flood event, and for India

**Question.** Every accuracy figure so far averages eleven flood events on six continents. What does the shipped radar rule score on **Indian** floods, and how sure is that number?

**Protocol.** The shipped rule as it runs: mean(VV,VH) averaged to **200 m**, threshold read from `evaluation/spatial_results.json` (notebook 07's train-chosen value). The **India figure uses only valid + test chips** — the train chips helped choose the threshold. Its uncertainty range comes from resampling whole chips 2,000 times.

**What it cannot tell you.** Sen1Floods11 has one Indian event, in one year. A score on it describes that event's landscape, not every Indian flood — coastal, Himalayan and urban floods remain unmeasured.


In [1]:
!pip install rasterio numpy scipy


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
LOCAL = r"C:\sen1floods11"

import sys, json, importlib
from pathlib import Path

import numpy as np
import rasterio
from rasterio.warp import transform_bounds

root = Path(LOCAL)
EVALUATION = Path.cwd().resolve().parent.parent / "evaluation"
sys.path.insert(0, str(EVALUATION))

import spatial as SP, change as C, events as EV
for module in (SP, C, EV):
    importlib.reload(module)       # pick up edits without a kernel restart

spatial = json.loads((EVALUATION / "spatial_results.json").read_text(encoding="utf-8"))
DB_200 = spatial["step1_best_threshold_at_scale"]["200 m"]["db"]
print(f"shipped rule at 200 m: mean(VV,VH) < {DB_200} dB  (from notebook 07)")

s1 = {p.name.replace("_S1Hand.tif", ""): p for p in root.rglob("*_S1Hand.tif")}
labels = {p.name.replace("_LabelHand.tif", ""): p for p in root.rglob("*_LabelHand.tif")}
splits = SP.load_splits(root, hand_keys=set(labels))
split_of = {k: name for name, keys in splits.items() for k in keys}

shipped rule at 200 m: mean(VV,VH) < -18.5 dB  (from notebook 07)


In [3]:
chips = []
for key in sorted(k for k in split_of if k in s1 and k in labels):
    with rasterio.open(s1[key]) as src:
        vv = src.read(1).astype(np.float32); vh = src.read(2).astype(np.float32)
        lon0, lat0, lon1, lat1 = transform_bounds(src.crs, "EPSG:4326", *src.bounds)
    with rasterio.open(labels[key]) as src:
        truth = src.read(1).astype(np.int8)
    vv[vv < C.NODATA_FLOOR_DB] = np.nan; vh[vh < C.NODATA_FLOOR_DB] = np.nan
    truth = truth.copy(); truth[~(np.isfinite(vv) & np.isfinite(vh))] = -1
    if not (truth != -1).any():
        continue
    fused = C.fused(vv, vh)
    chips.append({"key": key, "split": split_of[key],
                  "fused200": SP.block_mean(fused, 20), "truth200": SP.block_label(truth, 20),
                  "centre": ((lon0 + lon1) / 2, (lat0 + lat1) / 2)})
print(f"{len(chips)} chips")

def counts200(chip):
    pred = np.isfinite(chip["fused200"]) & (chip["fused200"] < DB_200)
    return SP.counts(pred, chip["truth200"])

440 chips


## Every event, at 200 m

All chips per event, so these mix tuned and untuned chips — read them for the spread between landscapes, not as held-out scores.

In [4]:
per_event = EV.by_event(chips, counts200)
print(f"{'event':<12}{'IoU':>8}{'prec':>8}{'recall':>8}{'chips':>7}")
print("-" * 43)
for event, s in sorted(per_event.items(), key=lambda kv: -kv[1]["iou"]):
    print(f"{event:<12}{s['iou']:>8.3f}{s['precision']:>8.3f}{s['recall']:>8.3f}{s['chips']:>7}")

event            IoU    prec  recall  chips
-------------------------------------------
Mekong         0.835   0.990   0.842     30
Spain          0.776   0.880   0.868     30
Bolivia        0.718   0.913   0.770     15
Sri-Lanka      0.672   0.952   0.696     42
Nigeria        0.669   0.973   0.682     18
India          0.628   0.880   0.687     68
USA            0.526   0.590   0.828     69
Paraguay       0.473   0.728   0.574     66
Ghana          0.452   0.813   0.504     48
Pakistan       0.240   0.324   0.480     28
Somalia        0.163   0.217   0.393     26


## India, held out, with its uncertainty

In [5]:
india = EV.india_summary(chips, counts200)
h, a = india["held_out"], india["all"]
print(f"India, valid + test only : IoU {h['iou']:.3f}  (95% range {h['ci95'][0]:.3f} to {h['ci95'][1]:.3f})  "
      f"precision {h['precision']:.3f}  recall {h['recall']:.3f}  {h['chips']} chips")
print(f"India, all chips         : IoU {a['iou']:.3f}  ({a['chips']} chips, includes chips used for tuning)")

centres = np.array([c["centre"] for c in chips if EV.event_of(c["key"]) == "India"])
if len(centres):
    print(f"\nIndian chips lie between {centres[:,1].min():.2f}-{centres[:,1].max():.2f} N and "
          f"{centres[:,0].min():.2f}-{centres[:,0].max():.2f} E (centre {centres[:,1].mean():.2f} N, "
          f"{centres[:,0].mean():.2f} E). Name the state from these before quoting the figure.")

India, valid + test only : IoU 0.725  (95% range 0.564 to 0.829)  precision 0.902  recall 0.787  28 chips
India, all chips         : IoU 0.628  (68 chips, includes chips used for tuning)

Indian chips lie between 25.73-27.30 N and 92.42-93.94 E (centre 26.47 N, 93.15 E). Name the state from these before quoting the figure.


In [6]:
block = {
    "event": "Sen1Floods11 India event (2016)",
    "chips": h["chips"],
    "iou": round(h["iou"], 3),
    "precision": round(h["precision"], 3),
    "recall": round(h["recall"], 3),
    "ci95": [round(h["ci95"][0], 3), round(h["ci95"][1], 3)],
}
payload = {
    "rule": {"band": "mean(VV,VH)", "scale_m": 200, "threshold_db": DB_200},
    "per_event": per_event,
    "india": india,
    "india_centre_lonlat": centres.mean(axis=0).tolist() if len(centres) else None,
    "india_validation_block_for_backend": block,
}
out = EVALUATION / "events_results.json"
out.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"written to {out}\n")
print("Add this to SCALE_RULES[200]['validation'] in backend/detection/sar.py as \"india\":\n")
print(json.dumps(block, indent=4))

written to C:\Users\punit\OneDrive\Desktop\Group_project\application\antardrishti\evaluation\events_results.json

Add this to SCALE_RULES[200]['validation'] in backend/detection/sar.py as "india":

{
    "event": "Sen1Floods11 India event (2016)",
    "chips": 28,
    "iou": 0.725,
    "precision": 0.902,
    "recall": 0.787,
    "ci95": [
        0.564,
        0.829
    ]
}


## What to do with the number

Add the printed block to `SCALE_RULES[200]["validation"]` in `backend/detection/sar.py` under the key `"india"`, replacing `"Sen1Floods11 India event (2016)"` with the state the coordinates point to. Every 200 m flood result then ends its accuracy sentence with *"On held-out Indian chips (N chips, …) it scores IoU X (95% range a to b)."* A test ties the figure to `events_results.json`.

**If the range is wide** — say, wider than 0.2 — quote the range, not the point. That width is the honest answer to "how accurate is this in India?" with the data that exists.
